# Notebook 10 — QDA (Reviewer-Response Version)

*Quadratic Discriminant Analysis*

**Prerequisites:** Run `00_Setup_PCA300_ReviewerResponse.ipynb` first.

| Fix | What / which comment |
|---|---|
| **Critical: wrong dataset path** | Same `Algerian_Lamiaceae` (missing `_v2`) bug as AdaBoost/LDA. Fixed, plus the same hard shape assertion. |
| **Critical bug** | Same unfitted-model-pickle issue. Fixed identically: `evaluate()` fits and returns the actual pickled object, with a reload smoke-test. |
| comment 10 | No hyperparameter search existed (default `reg_param=0.0`, i.e. unregularized per-class covariance estimation). Added: Optuna over `reg_param`, tuned on the held-out validation split. |
| comment 9 | AUC methodology stated: QuadraticDiscriminantAnalysis's native `predict_proba` used directly. |
| comment 7, 8, Q3, Q4 | Same bootstrap 95% CIs, per-class report, confusion matrix, small-support warning, and pickle smoke-test as every other classifier notebook. |

**Note for Methods**: unlike LDA, QDA estimates a separate 300x300
covariance matrix per class (no shared-covariance assumption). With 5,000
SMOTE-resampled training samples per class this is well-conditioned in
principle, but `reg_param` regularization is still worth tuning to guard
against near-singular estimates in any class with less internal variance.
QDA does not support `class_weight` (not a parameter it accepts).

**This notebook is not executed in this environment** -- built to run locally
against your existing feature caches and split file.


## Step 0 — Setup

In [1]:
import os, warnings, pickle, copy, json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)

warnings.filterwarnings("ignore")
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

# FIX: was "Algerian_Lamiaceae" (missing _v2).
DATASET_ROOT = "/home/ali/Desktop/Algerian_Lamiaceae_v3"
OUTPUT_DIR   = os.path.join(DATASET_ROOT, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")


Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Output: /home/ali/Desktop/Algerian_Lamiaceae_v3/results


## Step 1 — Load Cached Features and Train/Val/Test Split

In [2]:
HC_CACHE    = os.path.join(OUTPUT_DIR, "hc_features_cache.npz")
DEEP_CACHE  = os.path.join(OUTPUT_DIR, "deep_features_cache.npz")
SPLIT_CACHE = os.path.join(OUTPUT_DIR, "train_test_split.npz")

X_hc = np.load(HC_CACHE, allow_pickle=True)["X"]
print(f"Handcrafted features: {X_hc.shape}")

if os.path.exists(DEEP_CACHE):
    X_deep = np.load(DEEP_CACHE, allow_pickle=True)["X"]
    X_all  = np.concatenate([X_deep, X_hc], axis=1)
    print(f"Deep features: {X_deep.shape}")
else:
    X_all = X_hc
    print("Deep features not found -- using handcrafted only")
print(f"Fused features: {X_all.shape}")

split = np.load(SPLIT_CACHE, allow_pickle=True)
X_train_sc = split["X_train_sc"]
X_val_sc   = split["X_val_sc"]
X_test_sc  = split["X_test_sc"]
y_train_sm = split["y_train_sm"]
y_val_raw  = split["y_val_raw"]
y_test_raw = split["y_test_raw"]
n_components = int(split["n_components"])

with open(os.path.join(OUTPUT_DIR, "label_encoder_species.pkl"), "rb") as f:
    le = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "sc1.pkl"), "rb") as f:
    sc1 = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "sc2.pkl"), "rb") as f:
    sc2 = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "pca.pkl"), "rb") as f:
    pca = pickle.load(f)

class_names = list(le.classes_)
NUM_CLASSES = len(class_names)

print(f"Train: {X_train_sc.shape}  Val: {X_val_sc.shape}  Test: {X_test_sc.shape}")
print(f"Classes: {NUM_CLASSES} -- {class_names}")



Handcrafted features: (19145, 73)
Deep features: (19145, 1536)
Fused features: (19145, 1609)
Train: (35000, 300)  Val: (2372, 300)  Test: (2366, 300)
Classes: 7 -- [np.str_('Lamium_pooled'), np.str_('Lavandula_angustifolia'), np.str_('Lavandula_pooled'), np.str_('Leucas_aspera'), np.str_('Mentha_pooled'), np.str_('Ocimum_pooled'), np.str_('Plectranthus_pooled')]


## Step 2 — Evaluation Utilities (identical to previous classifiers)

In [3]:
N_BOOTSTRAP = 1000
SMALL_SUPPORT_THRESHOLD = 30

def evaluate(clf, Xtr, Xte, ytr, yte):
    clf.fit(Xtr, ytr)
    yp = clf.predict(Xte)
    yproba = None
    auc_s = None
    if hasattr(clf, "predict_proba"):
        try:
            yproba = clf.predict_proba(Xte)
            auc_s = round(roc_auc_score(yte, yproba, multi_class="ovr",
                                         average="macro") * 100, 2)
        except Exception:
            pass
    return {
        "Accuracy":  round(accuracy_score(yte, yp) * 100, 2),
        "Precision": round(precision_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "Recall":    round(recall_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "F1":        round(f1_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "AUC": auc_s,
        "_yp": yp, "_yproba": yproba, "_fitted_clf": clf,
    }

def bootstrap_ci(y_true, y_pred, y_proba=None, n_boot=N_BOOTSTRAP, seed=GLOBAL_SEED):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true)
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        accs.append(accuracy_score(yt, yp))
        precs.append(precision_score(yt, yp, average="weighted", zero_division=0))
        recs.append(recall_score(yt, yp, average="weighted", zero_division=0))
        f1s.append(f1_score(yt, yp, average="weighted", zero_division=0))
        if y_proba is not None:
            try:
                aucs.append(roc_auc_score(yt, y_proba[idx], multi_class="ovr", average="macro"))
            except ValueError:
                pass

    def summarize(arr):
        arr = np.array(arr) * 100
        return {"mean": round(float(arr.mean()), 2),
                "ci_lo": round(float(np.percentile(arr, 2.5)), 2),
                "ci_hi": round(float(np.percentile(arr, 97.5)), 2)}

    result = {"Accuracy": summarize(accs), "Precision": summarize(precs),
              "Recall": summarize(recs), "F1": summarize(f1s)}
    if y_proba is not None:
        result["AUC"] = summarize(aucs) if aucs else None
        result["AUC_valid_resamples"] = len(aucs)
        result["AUC_skipped_resamples"] = n_boot - len(aucs)
    return result

def per_class_report(y_true, y_pred, class_names, small_support_threshold=SMALL_SUPPORT_THRESHOLD):
    report = classification_report(y_true, y_pred, labels=list(range(len(class_names))),
                                    target_names=class_names, output_dict=True, zero_division=0)
    rows = []
    for cls in class_names:
        r = report[cls]
        rows.append({"Class": cls, "Precision": round(r["precision"] * 100, 2),
                     "Recall": round(r["recall"] * 100, 2),
                     "F1": round(r["f1-score"] * 100, 2), "Support": int(r["support"])})
    df = pd.DataFrame(rows)
    small = df[df["Support"] < small_support_threshold]
    if not small.empty:
        print(f"  \u26a0 Small test support (<{small_support_threshold} samples) -- "
              f"per-class scores below should not be over-interpreted (comment 7):")
        for _, r in small.iterrows():
            print(f"     {r['Class']:<28} support={r['Support']:>4}  "
                  f"P={r['Precision']:.1f}%  R={r['Recall']:.1f}%  F1={r['F1']:.1f}%")
    return df

def save_full_result(name, m, y_true, class_names):
    clf = m["_fitted_clf"]
    yp, yproba = m["_yp"], m["_yproba"]

    ci = bootstrap_ci(y_true, yp, yproba)
    per_class = per_class_report(y_true, yp, class_names)
    cm = confusion_matrix(y_true, yp, labels=list(range(len(class_names))))

    result = {k: v for k, v in m.items() if not k.startswith("_")}
    result["Classifier"] = name
    result["Bootstrap_CI_95"] = ci
    result["N_bootstrap"] = N_BOOTSTRAP
    result["PerClass"] = per_class.to_dict(orient="records")

    out_path = os.path.join(OUTPUT_DIR, f"result_{name.replace(' ','_')}.json")
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)

    mdl_path = os.path.join(OUTPUT_DIR, f"model_{name.replace(' ','_')}.pkl")
    with open(mdl_path, "wb") as f:
        pickle.dump(clf, f)

    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{name.replace(' ','_')}.csv")
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_path)

    auc_s = f"{m['AUC']}%" if m["AUC"] is not None else "n/a"
    print(f"  Acc={m['Accuracy']:6.2f}% [{ci['Accuracy']['ci_lo']:.1f}-{ci['Accuracy']['ci_hi']:.1f}]  "
          f"P={m['Precision']:6.2f}%  R={m['Recall']:6.2f}%  F1={m['F1']:6.2f}% "
          f"[{ci['F1']['ci_lo']:.1f}-{ci['F1']['ci_hi']:.1f}]  AUC={auc_s}")
    print(f"  Saved -> {out_path}")
    print(f"  Saved -> {mdl_path}")
    print(f"  Saved -> {cm_path}")

    with open(mdl_path, "rb") as f:
        reloaded = pickle.load(f)
    _ = reloaded.predict(X_test_sc[:1])
    print(f"  Smoke-test passed: reloaded model predicts without error.")

    return result

print("Evaluation utilities ready (fit-and-return, bootstrap CI, per-class, confusion matrix).")


Evaluation utilities ready (fit-and-return, bootstrap CI, per-class, confusion matrix).


## Step 3 — Baseline QDA (reg_param=0.0, matches original notebook)

In [4]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

name = "QDA"
print(f"Training {name} ...")
print("AUC methodology: QuadraticDiscriminantAnalysis's native predict_proba "
      "used directly. AUC is One-vs-Rest, macro-averaged.")

clf = QuadraticDiscriminantAnalysis()

m = evaluate(clf, X_train_sc, X_test_sc, y_train_sm, y_test_raw)
result_baseline = save_full_result(name, m, y_test_raw, class_names)


Training QDA ...
AUC methodology: QuadraticDiscriminantAnalysis's native predict_proba used directly. AUC is One-vs-Rest, macro-averaged.
  ⚠ Small test support (<30 samples) -- per-class scores below should not be over-interpreted (comment 7):
     Leucas_aspera                support=  11  P=100.0%  R=90.9%  F1=95.2%
  Acc= 89.69% [88.5-90.9]  P= 90.16%  R= 89.69%  F1= 89.85% [88.6-91.0]  AUC=97.73%
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/result_QDA.json
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/model_QDA.pkl
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/confusion_matrix_QDA.csv
  Smoke-test passed: reloaded model predicts without error.


## Step 4 — Optuna Hyperparameter Search (new; comment 10)

Tuned on the held-out **validation** split; Test is not touched here.


In [5]:
import optuna
from optuna.samplers import TPESampler

N_TRIALS = 40
SEARCH_SPACE_DESC = "reg_param ~ Uniform[0.0, 1.0]"
print(f"Optuna search space (comment 10): {SEARCH_SPACE_DESC}")
print(f"seed={GLOBAL_SEED}, n_trials={N_TRIALS}, sampler=TPESampler")
print("Objective: fit on SMOTE-resampled TRAIN, score weighted-F1 on untouched VAL.")

def objective_qda(trial):
    reg_param = trial.suggest_float("reg_param", 0.0, 1.0)
    clf = QuadraticDiscriminantAnalysis(reg_param=reg_param)
    clf.fit(X_train_sc, y_train_sm)
    yp = clf.predict(X_val_sc)
    return f1_score(y_val_raw, yp, average="weighted")

study_qda = optuna.create_study(direction="maximize", sampler=TPESampler(seed=GLOBAL_SEED))
study_qda.optimize(objective_qda, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest VAL F1(weighted) = {study_qda.best_value*100:.2f}%")
print(f"Best params: {study_qda.best_params}")

optuna_best_qda = {
    "Classifier": "QDA_Optuna_BestTrial",
    "Val_F1_weighted": round(study_qda.best_value * 100, 2),
    "BestParams": study_qda.best_params,
    "NumTrials": N_TRIALS,
    "SearchSpace": SEARCH_SPACE_DESC,
    "Sampler": "TPESampler",
    "Seed": GLOBAL_SEED,
    "TuningSet": "validation (held out, not SMOTE-resampled)",
}
with open(os.path.join(OUTPUT_DIR, "optuna_result_QDA.json"), "w") as f:
    json.dump(optuna_best_qda, f, indent=2)
print(optuna_best_qda)


[I 2026-10-18 04:07:03,824] A new study created in memory with name: no-name-f8dc6fb3-541d-4c76-bfdb-5d591561b653


Optuna search space (comment 10): reg_param ~ Uniform[0.0, 1.0]
seed=42, n_trials=40, sampler=TPESampler
Objective: fit on SMOTE-resampled TRAIN, score weighted-F1 on untouched VAL.


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-10-18 04:09:16,754] Trial 0 finished with value: 0.9077000722174455 and parameters: {'reg_param': 0.3745401188473625}. Best is trial 0 with value: 0.9077000722174455.
[I 2026-10-18 04:11:25,301] Trial 1 finished with value: 0.9102169901494874 and parameters: {'reg_param': 0.9507143064099162}. Best is trial 1 with value: 0.9102169901494874.
[I 2026-10-18 04:13:04,008] Trial 2 finished with value: 0.9078923906006968 and parameters: {'reg_param': 0.7319939418114051}. Best is trial 1 with value: 0.9102169901494874.
[I 2026-10-18 04:14:43,292] Trial 3 finished with value: 0.9062578624273412 and parameters: {'reg_param': 0.5986584841970366}. Best is trial 1 with value: 0.9102169901494874.
[I 2026-10-18 04:16:19,564] Trial 4 finished with value: 0.9103245470969065 and parameters: {'reg_param': 0.15601864044243652}. Best is trial 4 with value: 0.9103245470969065.
[I 2026-10-18 04:17:57,144] Trial 5 finished with value: 0.9103245470969065 and parameters: {'reg_param': 0.1559945203362026

## Step 5 — Final Model: Retrain with Best Hyperparameters, Evaluate Once on Test

In [6]:
name = "QDA_Optuna"
print(f"Training {name} (final retrained model, evaluated ONCE on held-out test) ...")

clf_final = QuadraticDiscriminantAnalysis(reg_param=study_qda.best_params["reg_param"])

m_final = evaluate(clf_final, X_train_sc, X_test_sc, y_train_sm, y_test_raw)
result_optuna = save_full_result(name, m_final, y_test_raw, class_names)

print("\n" + "="*70)
print("BASELINE (reg_param=0.0) vs. OPTUNA-TUNED -- compare on TEST:")
print("="*70)
print(f"  Baseline : Acc={result_baseline['Accuracy']:.2f}%  F1={result_baseline['F1']:.2f}%  "
      f"AUC={result_baseline['AUC']}%")
print(f"  Optuna   : Acc={result_optuna['Accuracy']:.2f}%  F1={result_optuna['F1']:.2f}%  "
      f"AUC={result_optuna['AUC']}%")


Training QDA_Optuna (final retrained model, evaluated ONCE on held-out test) ...
  ⚠ Small test support (<30 samples) -- per-class scores below should not be over-interpreted (comment 7):
     Leucas_aspera                support=  11  P=100.0%  R=100.0%  F1=100.0%
  Acc= 89.56% [88.2-90.7]  P= 90.43%  R= 89.56%  F1= 89.87% [88.7-91.0]  AUC=98.5%
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/result_QDA_Optuna.json
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/model_QDA_Optuna.pkl
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/confusion_matrix_QDA_Optuna.csv
  Smoke-test passed: reloaded model predicts without error.

BASELINE (reg_param=0.0) vs. OPTUNA-TUNED -- compare on TEST:
  Baseline : Acc=89.69%  F1=89.85%  AUC=97.73%
  Optuna   : Acc=89.56%  F1=89.87%  AUC=98.5%
